In [ ]:
import event_helpers
import gcsfs
import geopandas as gpd
import google.auth
import pandas as pd
import poi_geography_utils as poi_geog_utils
import world_cup_vars as wc_vars
from gtfs_curator_utils import utils

GCS_FILE_PATH = wc_vars.GCS_FILE_PATH
credentials, _ = google.auth.default()

In [ ]:
STADIUM_NAME = "sofi"
stop_gdf = gpd.read_parquet(
    f"{GCS_FILE_PATH}stop_summary_{STADIUM_NAME}.parquet",
    storage_options = {"token": credentials.token}
)

In [ ]:
import create_route_and_stop_event_summary as A1

routes_df = A1.import_routes_near_poi(wc_vars.event_name, "sofi")

stop_gdf = A1.prep_fct_daily_scheduled_stops(wc_vars.event_name, wc_vars.sofi_match_times).merge(
    routes_df[["feed_key", "schedule_name"]].drop_duplicates(),
    how="inner",
)

In [ ]:
# get non-event 
arrivals_cols = [c for c in stop_gdf.columns if "arrivals_per_hour" in c]

nonevent_arrivals_df = (
    event_helpers.aggregate_by_event_type(
        stop_gdf[~stop_gdf.service_date.isin(wc_vars.sofi_dates)],
        group_cols=[
            "schedule_name",
            "stop_id",
            "stop_name",
            "event_day",
            "day_type",
        ],
        mean_cols=arrivals_cols
    )
)

In [ ]:
arrivals_by_event = pd.concat(
    [nonevent_arrivals_df,
     stop_gdf[stop_gdf.service_date.isin(wc_vars.sofi_dates)]
    ], axis=0, ignore_index=True
)

In [ ]:
arrivals_by_event_long = event_helpers.make_long(
    arrivals_by_event, 
    id_vars = ["service_date", "schedule_name", "stop_id", "stop_name", "day_type"] # service_date for non_event are NaT
)

In [ ]:
# Aggregate this across number of stops for operator
# Keep average arrivals_per_hour
arrivals_by_event_long2 = (
    arrivals_by_event_long
    .groupby(["service_date", "schedule_name", "time_of_day", "day_type"], dropna=False)
    .agg({
        "arrivals_per_hour": "mean",
        "stop_id": "count"
    }).reset_index().rename(columns = {"stop_id": "n_stops"})
)

arrivals_by_event_long2 = arrivals_by_event_long2.assign(
    service_date = arrivals_by_event_long2.service_date.dt.date.fillna('nonevent').astype(str),
)

In [ ]:
import altair as alt

alt.data_transformers.enable("vegafusion")

def bar_chart_by_time_of_day(df: pd.DataFrame):
    chart = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x="arrivals_per_hour",
            y=alt.Y("time_of_day", sort=event_helpers.TIME_OF_DAY_ORDER),
            tooltip = ["schedule_name", "day_type", "time_of_day", "arrivals_per_hour", "n_stops"],
            color="schedule_name:N"
        )
    )
    
    return chart

### Sofi weekday midday matches

In [ ]:
time_of_day = "midday"
day_type = "weekday"

filtered_sofi_dict = event_helpers.grab_matches_by_day_type(
    wc_vars.sofi_match_times, time_of_day, day_type)
filtered_sofi_dict

In [ ]:
arrivals_by_event_long2.dtypes

In [ ]:
bar_chart_by_time_of_day(
    arrivals_by_event_long2[arrivals_by_event_long2.service_date == "2026-06-18"]
)

In [ ]:
bar_chart_by_time_of_day(
    arrivals_by_event_long2[arrivals_by_event_long2.service_date == "nonevent"]
)

In [ ]:
alt.vconcat(
    non_event_chart, event_jun18_chart, 
    event_jul2_chart, event_jul10_chart
).resolve_scale(x="shared")